# Breakthrough notebook: stability-aware blended models for stronger grouped CV

In [1]:
# ============================================================
# Cell A1. Imports + Global Config
# ============================================================

import os
import re
import json
import math
import warnings
from copy import deepcopy
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import joblib
import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.utils import column_index_from_string, get_column_letter

from scipy.stats import spearmanr, loguniform, uniform, randint

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression

from sklearn.model_selection import GroupKFold, GroupShuffleSplit, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import Ridge, ElasticNet, ElasticNetCV, BayesianRidge, HuberRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

# ----------------------------
# Raw import config
# ----------------------------
IMPORT_PATH = r"C:\Users\김민겸\Desktop\Metal metamaterial\Training\Total data_260503.xlsx"

RUN_TAG = datetime.now().strftime("%y%m%d_%H%M%S")
EXPORT_ROOT_NAME = f"Result_enhanced_featureaware_{RUN_TAG}"
EXPORT_ROOT_DIR = os.path.join(os.path.dirname(IMPORT_PATH), EXPORT_ROOT_NAME)

RAW_EXPORT_DIR = os.path.join(EXPORT_ROOT_DIR, "01_RawData")
MODEL_EXPORT_DIR = os.path.join(EXPORT_ROOT_DIR, "02_ModelComparison")
FINAL_BACKUP_DIR = os.path.join(EXPORT_ROOT_DIR, "03_FinalModelBackup")
FINAL_DATA_DIR = os.path.join(EXPORT_ROOT_DIR, "04_FinalSelectedDatasets")
PER_OUTPUT_DIR = os.path.join(MODEL_EXPORT_DIR, "per_output")

HEADER_MAIN_ROW = 2
HEADER_SUB_ROWS = [3, 4, 5]

DATA_ROW_START = 7
DATA_ROW_END = 204

RANGE_INPUT  = ("I",  DATA_ROW_START, "FU", DATA_ROW_END)
RANGE_OUTPUT = ("FV", DATA_ROW_START, "HL", DATA_ROW_END)

OUTPUT_COLUMNS = ["FW", "FX", "FZ", "GA", "GC", "GG", "GJ", "GZ", "HA", "HB", "HC", "HD", "HE", "HG", "HI", "HK"]

EXPORT_FORMAT_XLSX = True
EXPORT_FORMAT_CSV  = True

# ----------------------------
# Dataset config
# ----------------------------
ROUND_X_DECIMALS = 8
MIN_GROUPS_FOR_MODEL = 12
TEST_OUTPUTS = None
# 예시: TEST_OUTPUTS = ["Modulus", "Yield strength"]

# ----------------------------
# CV config
# ----------------------------
RANDOM_STATE = 42
FINAL_OUTER_SPLITS = 5
OUTER_REPEATS = 12
OUTER_TEST_SIZE = 0.22
INNER_GSS_SPLITS = 24
INNER_GSS_TEST_SIZE = 0.20
SEARCH_N_JOBS = 1

# ----------------------------
# Feature/model config
# ----------------------------
ROUGH_PREFILTER_TOPK = 20
ROUGH_PREFILTER_TOPK_BY_FAMILY = {
    "mechanical_energy": 16,
    "thermal": 20,
    "vibrational": 22,
}
FINAL_TOPK_CANDIDATES = [2, 3, 4, 6]
TOPK_BY_FAMILY = {
    "mechanical_energy": [2, 3, 4],
    "thermal": [2, 3, 4, 6],
    "vibrational": [2, 3, 4, 6],
}
MAX_FINAL_FEATURES = 6
CORR_PRUNE_THRESHOLD = 0.85

FEATURE_SCORE_WEIGHTS = {
    "spearman": 0.30,
    "pearson": 0.15,
    "mutual_info": 0.15,
    "elastic_net": 0.40,
}

MODEL_AWARE_SCORING_MODELS = ["Bayesian_Ridge", "Ridge", "PLS_Regression"]
FINAL_MODEL_NAMES = [
    "Bayesian_Ridge",
    "Ridge",
    "ElasticNet_CV",
    "Huber_Regressor",
    "PLS_Regression",
    "PCR_Ridge",
    "Kernel_Ridge_RBF",
    "SVR_RBF",
]

MODEL_COMPLEXITY_RANK = {
    "Bayesian_Ridge": 1,
    "Ridge": 2,
    "ElasticNet_CV": 3,
    "Huber_Regressor": 4,
    "PLS_Regression": 5,
    "PCR_Ridge": 6,
    "Kernel_Ridge_RBF": 7,
    "SVR_RBF": 8,
    "WeightedBlend_2": 9,
    "WeightedBlend_3": 10,
}

TARGET_TRANSFORM_CANDIDATES = ["raw", "yeo_johnson"]

INNER_SCORING_BY_FAMILY = {
    "mechanical_energy": "r2",
    "thermal": "r2",
    "vibrational": "r2",
}
SUMMARY_STD_PENALTY = 0.10
SUMMARY_SUPPORT_BONUS = 0.12
MIN_SUPPORT_SHARE_BY_FAMILY = {
    "mechanical_energy": 0.15,
    "thermal": 0.15,
    "vibrational": 0.20,
}

SELECTION_TOLERANCE_FRAC = 0.03
MIN_TRAIN_R2_GATE = 0.01
CONSENSUS_MIN_FOLD_SHARE = 0.40
ROBUST_TRIM_Z = 1.5

# ----------------------------
# Conservative feature engineering config
# ----------------------------
ENGINEERED_FEATURES_ENABLED = True
ENGINEERED_MAX_BASE_BY_FAMILY = {
    "mechanical_energy": 4,
    "thermal": 5,
    "vibrational": 5,
}
ENGINEERED_INCLUDE_SIGNED_LOG = True
ENGINEERED_INCLUDE_SQUARE = True
ENGINEERED_INCLUDE_PRODUCT_BY_FAMILY = {
    "mechanical_energy": True,
    "thermal": True,
    "vibrational": True,
}
ENGINEERED_INCLUDE_RATIO_BY_FAMILY = {
    "mechanical_energy": True,
    "thermal": True,
    "vibrational": False,
}
ENGINEERED_MAX_TOTAL_ADDED = 18

# repeated-Y handling
APPLY_REPEATED_Y_ONLY_IF_NEEDED = True

# selection objective
TOPK_PENALTY = 0.010
COMPLEXITY_PENALTY = 0.005
MIN_VALID_EVAL_SAMPLES = 3



os.makedirs(RAW_EXPORT_DIR, exist_ok=True)
os.makedirs(MODEL_EXPORT_DIR, exist_ok=True)
os.makedirs(FINAL_BACKUP_DIR, exist_ok=True)
os.makedirs(FINAL_DATA_DIR, exist_ok=True)
os.makedirs(PER_OUTPUT_DIR, exist_ok=True)

print("IMPORT_PATH      :", IMPORT_PATH)
print("EXPORT_ROOT_DIR  :", EXPORT_ROOT_DIR)
print("FINAL_BACKUP_DIR :", FINAL_BACKUP_DIR)
print("FINAL_DATA_DIR   :", FINAL_DATA_DIR)
print("FINAL_MODEL_NAMES:", FINAL_MODEL_NAMES)
print("OUTER_REPEATS    :", OUTER_REPEATS)
print("TOPK_BY_FAMILY   :", TOPK_BY_FAMILY)

# ----------------------------
# Blend / robust search config
# ----------------------------
BLEND_TOP_CANDIDATES = [2, 3]
BLEND_MIN_BASE_SCORE = 0.0
BLEND_EVAL_FOLDS = 5
TRAIN_WINSOR_CLIP = 0.02


IMPORT_PATH      : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Total data_260503.xlsx
EXPORT_ROOT_DIR  : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Result_enhanced_featureaware_260508_104111
FINAL_BACKUP_DIR : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Result_enhanced_featureaware_260508_104111\03_FinalModelBackup
FINAL_DATA_DIR   : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Result_enhanced_featureaware_260508_104111\04_FinalSelectedDatasets
FINAL_MODEL_NAMES: ['Bayesian_Ridge', 'Ridge', 'ElasticNet_CV', 'Huber_Regressor', 'PLS_Regression', 'PCR_Ridge', 'Kernel_Ridge_RBF', 'SVR_RBF']
OUTER_REPEATS    : 12
TOPK_BY_FAMILY   : {'mechanical_energy': [2, 3, 4], 'thermal': [2, 3, 4, 6], 'vibrational': [2, 3, 4, 6]}


In [2]:
# ============================================================
# Cell A2. Helper Functions - Excel / Header / Export
# ============================================================

def _ffill_horiz(values):
    out = []
    last = None
    for v in values:
        if v is None or str(v).strip() == "":
            out.append(last)
        else:
            last = str(v).strip()
            out.append(last)
    return out

def _sanitize_header(text):
    text = "" if text is None else str(text)
    text = text.replace("\n", " ").replace("\r", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text

def _make_unique(names):
    seen = Counter()
    out = []
    for n in names:
        base = n if n else "Unnamed"
        seen[base] += 1
        out.append(base if seen[base] == 1 else f"{base}__{seen[base]}")
    return out

def build_headers_from_rows(ws, start_col_letter, end_col_letter, main_row, sub_rows):
    start_col = column_index_from_string(start_col_letter)
    end_col = column_index_from_string(end_col_letter)
    col_indices = list(range(start_col, end_col + 1))

    all_rows = [main_row] + list(sub_rows)
    row_values = []
    for r in all_rows:
        vals = [ws.cell(row=r, column=c).value for c in col_indices]
        row_values.append(_ffill_horiz(vals))

    headers = []
    for i, col_idx in enumerate(col_indices):
        parts = []
        for row_vals in row_values:
            v = row_vals[i]
            if v is not None and str(v).strip() != "":
                v = _sanitize_header(v)
                if len(parts) == 0 or parts[-1] != v:
                    parts.append(v)
        if len(parts) == 0:
            parts = [f"COL_{get_column_letter(col_idx)}"]
        headers.append(" | ".join(parts))

    return _make_unique(headers), [get_column_letter(c) for c in col_indices]

def extract_range_df(ws, range_spec, header_main_row, header_sub_rows):
    start_col, start_row, end_col, end_row = range_spec
    headers, letters = build_headers_from_rows(
        ws=ws,
        start_col_letter=start_col,
        end_col_letter=end_col,
        main_row=header_main_row,
        sub_rows=header_sub_rows
    )
    start_idx = column_index_from_string(start_col)
    end_idx = column_index_from_string(end_col)

    data = []
    for r in range(start_row, end_row + 1):
        row_vals = [ws.cell(row=r, column=c).value for c in range(start_idx, end_idx + 1)]
        data.append(row_vals)

    df = pd.DataFrame(data, columns=headers)
    df.attrs["excel_letters"] = letters
    return df

def export_df(df, path_no_ext, index=False):
    if EXPORT_FORMAT_CSV:
        df.to_csv(path_no_ext + ".csv", index=index, encoding="utf-8-sig")
    if EXPORT_FORMAT_XLSX:
        df.to_excel(path_no_ext + ".xlsx", index=index)

def _to_jsonable(obj):
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, np.ndarray):
        return [_to_jsonable(x) for x in obj.tolist()]
    if isinstance(obj, pd.Series):
        if obj.index.is_unique:
            return {str(k): _to_jsonable(v) for k, v in obj.to_dict().items()}
        return [_to_jsonable(x) for x in obj.tolist()]
    if isinstance(obj, pd.DataFrame):
        return [{str(k): _to_jsonable(v) for k, v in row.items()} for row in obj.to_dict(orient="records")]
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_to_jsonable(x) for x in obj]
    if hasattr(obj, "item"):
        try:
            return _to_jsonable(obj.item())
        except Exception:
            pass
    return str(obj)

def export_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_to_jsonable(obj), f, ensure_ascii=False, indent=2)

def safe_numeric_df(df):
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def sanitize_filename(text, max_len=120):
    text = re.sub(r'[\\/:*?"<>|]+', "_", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len]

In [3]:
# ============================================================
# Cell B1. Workbook Load + Raw Range Extraction
# ============================================================

wb = load_workbook(IMPORT_PATH, data_only=True)
sheet_name = wb.sheetnames[0]
ws = wb[sheet_name]

input_raw_df = extract_range_df(ws, RANGE_INPUT, HEADER_MAIN_ROW, HEADER_SUB_ROWS)
output_raw_all_df = extract_range_df(ws, RANGE_OUTPUT, HEADER_MAIN_ROW, HEADER_SUB_ROWS)

output_letter_to_name = dict(zip(output_raw_all_df.attrs["excel_letters"], output_raw_all_df.columns))
selected_output_names = [output_letter_to_name[c] for c in OUTPUT_COLUMNS if c in output_letter_to_name]
output_raw_df = output_raw_all_df[selected_output_names].copy()

export_df(input_raw_df, os.path.join(RAW_EXPORT_DIR, "input_raw"))
export_df(output_raw_df, os.path.join(RAW_EXPORT_DIR, "output_raw_selected"))

print("Loaded workbook :", sheet_name)
print("Input shape     :", input_raw_df.shape)
print("Output shape    :", output_raw_df.shape)
print("Selected outputs:", len(selected_output_names))
display(pd.DataFrame({"output_excel_col": OUTPUT_COLUMNS, "output_name": selected_output_names}))

Loaded workbook : 총정리
Input shape     : (198, 169)
Output shape    : (198, 16)
Selected outputs: 16


,output_excel_col,output_name
0,FW,Modulus
1,FX,Com. Strength
2,FZ,APS
3,GA,AS
4,GC,Yield strength
5,GG,Densif. strength
6,GJ,Total energy
7,GZ,Thermal characteristics | Thermal conductivity...
8,HA,Thermal characteristics | h | W/m.K
9,HB,Thermal characteristics | Heating rate | °C/s


In [4]:
# ============================================================
# Cell B2. Numeric conversion + Same-X Group ID + DATA_BY_OUTPUT
# ============================================================

X_all = safe_numeric_df(input_raw_df)
Y_all = safe_numeric_df(output_raw_df)

X_all = X_all.loc[:, X_all.notna().any(axis=0)].copy()
Y_all = Y_all.loc[:, Y_all.notna().any(axis=0)].copy()

def build_group_ids_from_x(X_df, decimals=8):
    X_round = X_df.round(decimals)
    key_series = X_round.astype(str).agg("||".join, axis=1)
    codes, _ = pd.factorize(key_series, sort=False)
    return pd.Series(codes, name="GROUP_ID"), key_series.rename("X_KEY")

GROUP_ID, X_KEY = build_group_ids_from_x(X_all, decimals=ROUND_X_DECIMALS)

MASTER_DF = pd.concat([X_all, Y_all, GROUP_ID, X_KEY], axis=1)
group_size_df = GROUP_ID.value_counts().sort_index().rename("group_size").reset_index()
group_size_df.columns = ["GROUP_ID", "group_size"]

export_df(MASTER_DF, os.path.join(RAW_EXPORT_DIR, "master_numeric_with_group"))
export_df(group_size_df, os.path.join(RAW_EXPORT_DIR, "group_size_summary"))

DATA_BY_OUTPUT = {}
target_output_names = Y_all.columns.tolist()
if TEST_OUTPUTS is not None:
    target_output_names = [c for c in target_output_names if c in TEST_OUTPUTS]

summary_rows = []
for output_name in target_output_names:
    tmp = pd.concat([X_all, Y_all[[output_name]], GROUP_ID, X_KEY], axis=1)
    tmp = tmp.dropna(subset=[output_name]).reset_index(drop=True)
    if tmp.empty:
        continue

    DATA_BY_OUTPUT[output_name] = {
        "df": tmp.copy(),
        "feature_cols": X_all.columns.tolist(),
        "target_col": output_name,
        "group_col": "GROUP_ID",
        "xkey_col": "X_KEY",
    }

    summary_rows.append({
        "output_name": output_name,
        "n_rows": int(len(tmp)),
        "n_groups": int(tmp["GROUP_ID"].nunique()),
        "n_features": int(len(X_all.columns)),
    })

DATASET_SUMMARY_DF = pd.DataFrame(summary_rows).sort_values(["n_groups", "n_rows"], ascending=[False, False])
export_df(DATASET_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "dataset_summary_by_output"))

display(DATASET_SUMMARY_DF)
print("Prepared outputs:", len(DATA_BY_OUTPUT))

,output_name,n_rows,n_groups,n_features
12,Vibrational response | FRF (g/N) | 300-8000 Hz...,193,128,169
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,193,128,169
14,Vibrational response | FRF (g/N) | 3000-6500 H...,193,128,169
15,Vibrational response | FRF (g/N) | 6500-8000 H...,193,128,169
7,Thermal characteristics | Thermal conductivity...,65,65,169
9,Thermal characteristics | Heating rate | °C/s,65,65,169
10,Thermal characteristics | Cooling rate | °C/s,65,65,169
11,Thermal characteristics | Heating Temp | °C/s,65,65,169
0,Modulus,56,56,169
1,Com. Strength,56,56,169


Prepared outputs: 16


In [5]:
# ============================================================
# Cell C1. Repeated-group target variance diagnostics
# ============================================================

diag_rows = []

for output_name, bundle in DATA_BY_OUTPUT.items():
    df = bundle["df"].copy()
    target_col = bundle["target_col"]
    group_col = bundle["group_col"]

    # numeric safety
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df = df.dropna(subset=[target_col, group_col]).copy()

    if df.empty:
        continue

    grp = (
        df.groupby(group_col)[target_col]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index(drop=True)
    )

    grp["std"] = grp["std"].fillna(0.0)
    grp["range"] = (grp["max"] - grp["min"]).fillna(0.0)
    grp["cv_like"] = grp["std"] / (grp["mean"].abs() + 1e-12)
    grp["cv_like"] = grp["cv_like"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    diag_rows.append({
        "output_name": output_name,
        "n_groups": int(df[group_col].nunique()),
        "n_rows": int(len(df)),
        "median_group_size": float(df.groupby(group_col).size().median()),
        "median_group_std": float(grp["std"].median()),
        "mean_group_std": float(grp["std"].mean()),
        "median_group_range": float(grp["range"].median()),
        "mean_group_cv_like": float(grp["cv_like"].mean()),
    })

GROUP_VARIANCE_DIAG_DF = pd.DataFrame(diag_rows)

if not GROUP_VARIANCE_DIAG_DF.empty:
    GROUP_VARIANCE_DIAG_DF = GROUP_VARIANCE_DIAG_DF.sort_values(
        "mean_group_std", ascending=False
    ).reset_index(drop=True)

display(GROUP_VARIANCE_DIAG_DF)
export_df(
    GROUP_VARIANCE_DIAG_DF,
    os.path.join(MODEL_EXPORT_DIR, "repeated_group_target_variance_diag")
)

def classify_output_family(output_name):
    name = str(output_name)
    if "Thermal characteristics" in name:
        return "thermal"
    if "Vibrational response" in name:
        return "vibrational"
    return "mechanical_energy"

GROUP_VARIANCE_DIAG_DF["output_family"] = (
    GROUP_VARIANCE_DIAG_DF["output_name"].map(classify_output_family)
)

display(GROUP_VARIANCE_DIAG_DF)
export_df(
    GROUP_VARIANCE_DIAG_DF,
    os.path.join(MODEL_EXPORT_DIR, "target_variance_diagnostics")
)

,output_name,n_groups,n_rows,median_group_size,median_group_std,mean_group_std,median_group_range,mean_group_cv_like
0,Vibrational response | FRF (g/N) | 6500-8000 H...,128,193,1.0,0.0,0.099135,0.0,0.112132
1,Vibrational response | FRF (g/N) | 3000-6500 H...,128,193,1.0,0.0,0.037563,0.0,0.089067
2,Vibrational response | FRF (g/N) | 300-8000 Hz...,128,193,1.0,0.0,0.025187,0.0,0.068057
3,Vibrational response | FRF (g/N) | 300-3000 Hz...,128,193,1.0,0.0,0.006979,0.0,0.094633
4,Modulus,56,56,1.0,0.0,0.000000,0.0,0.000000
5,Com. Strength,56,56,1.0,0.0,0.000000,0.0,0.000000
6,APS,56,56,1.0,0.0,0.000000,0.0,0.000000
7,AS,56,56,1.0,0.0,0.000000,0.0,0.000000
8,Yield strength,56,56,1.0,0.0,0.000000,0.0,0.000000
9,Densif. strength,56,56,1.0,0.0,0.000000,0.0,0.000000


,output_name,n_groups,n_rows,median_group_size,median_group_std,mean_group_std,median_group_range,mean_group_cv_like,output_family
0,Vibrational response | FRF (g/N) | 6500-8000 H...,128,193,1.0,0.0,0.099135,0.0,0.112132,vibrational
1,Vibrational response | FRF (g/N) | 3000-6500 H...,128,193,1.0,0.0,0.037563,0.0,0.089067,vibrational
2,Vibrational response | FRF (g/N) | 300-8000 Hz...,128,193,1.0,0.0,0.025187,0.0,0.068057,vibrational
3,Vibrational response | FRF (g/N) | 300-3000 Hz...,128,193,1.0,0.0,0.006979,0.0,0.094633,vibrational
4,Modulus,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy
5,Com. Strength,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy
6,APS,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy
7,AS,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy
8,Yield strength,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy
9,Densif. strength,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy


In [6]:
# ============================================================
# Cell C2. Helper Functions - group splits / feature scores / model-aware Y selection
# ============================================================


def repeated_group_splits(groups, n_splits=5, n_repeats=8, random_state=42, return_split_id=False, test_size=None):
    groups = np.asarray(groups)
    unique_groups = np.array(pd.unique(groups))
    if len(unique_groups) < 2:
        return
    if test_size is None:
        test_size = OUTER_TEST_SIZE if "OUTER_TEST_SIZE" in globals() else 0.2

    split_id = 0
    if len(unique_groups) >= max(8, n_splits):
        gss = GroupShuffleSplit(
            n_splits=max(1, int(n_splits) * max(1, int(n_repeats))),
            test_size=test_size,
            random_state=random_state,
        )
        dummy_X = np.zeros((len(groups), 1))
        dummy_y = np.zeros(len(groups))
        for split_no, (tr_idx, va_idx) in enumerate(gss.split(dummy_X, dummy_y, groups=groups), start=1):
            split_id += 1
            repeat_id = int((split_no - 1) // max(1, n_splits) + 1)
            fold_id = int((split_no - 1) % max(1, n_splits) + 1)
            if return_split_id:
                yield split_id, repeat_id, fold_id, tr_idx, va_idx
            else:
                yield tr_idx, va_idx
    else:
        rng = np.random.RandomState(random_state)
        n_splits = min(n_splits, len(unique_groups))
        for rep in range(n_repeats):
            shuffled = unique_groups.copy()
            rng.shuffle(shuffled)
            folds = np.array_split(shuffled, n_splits)
            for fold_no, fold_groups in enumerate(folds, start=1):
                val_mask = np.isin(groups, fold_groups)
                tr_idx = np.where(~val_mask)[0]
                va_idx = np.where(val_mask)[0]
                if len(tr_idx) == 0 or len(va_idx) == 0:
                    continue
                split_id += 1
                if return_split_id:
                    yield split_id, rep + 1, fold_no, tr_idx, va_idx
                else:
                    yield tr_idx, va_idx


def aggregate_by_group_firstX(X_df, y, groups, y_method="median"):
    rows, ys, gs = [], [], []
    y = np.asarray(y, dtype=float)
    groups = np.asarray(groups)
    for g in pd.unique(groups):
        idx = np.where(groups == g)[0]
        block_X = X_df.iloc[idx]
        block_y = y[idx]
        rows.append(block_X.iloc[0].copy())
        if y_method == "mean":
            ys.append(float(np.nanmean(block_y)))
        else:
            ys.append(float(np.nanmedian(block_y)))
        gs.append(g)
    Xg = pd.DataFrame(rows).reset_index(drop=True)
    yg = pd.Series(ys, name="target").reset_index(drop=True)
    gg = pd.Series(gs, name="GROUP_ID").reset_index(drop=True)
    return Xg, yg, gg

def minmax_series(s):
    s = pd.Series(s, dtype=float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if len(s) == 0:
        return s
    vmin, vmax = float(s.min()), float(s.max())
    if abs(vmax - vmin) < 1e-15:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - vmin) / (vmax - vmin)

def corr_prune_from_ranked(X_df, ranked_features, keep_k=20, threshold=0.90):
    ranked_features = [f for f in ranked_features if f in X_df.columns]
    if len(ranked_features) <= 1:
        return ranked_features[:keep_k]

    corr = X_df[ranked_features].corr(method="spearman").abs().fillna(0.0)
    selected = []
    for feat in ranked_features:
        keep = True
        for chosen in selected:
            if corr.loc[feat, chosen] >= threshold:
                keep = False
                break
        if keep:
            selected.append(feat)
        if len(selected) >= keep_k:
            break
    return selected



def group_repeat_stats(groups):
    s = pd.Series(groups)
    counts = s.value_counts(dropna=False)
    if len(counts) == 0:
        return {"has_repeats": False, "max_group_size": 0, "median_group_size": 0.0}
    return {
        "has_repeats": bool((counts > 1).any()),
        "max_group_size": int(counts.max()),
        "median_group_size": float(counts.median()),
    }


def get_prefilter_topk_for_output(output_name):
    fam = classify_output_family(output_name)
    return int(ROUGH_PREFILTER_TOPK_BY_FAMILY.get(fam, ROUGH_PREFILTER_TOPK))

def get_inner_scoring_for_output(output_name):
    fam = classify_output_family(output_name)
    return INNER_SCORING_BY_FAMILY.get(fam, "r2")

def choose_best_summary_row(model_summary_df, output_name):
    fam = classify_output_family(output_name)
    if model_summary_df is None or len(model_summary_df) == 0:
        return None
    df = model_summary_df.copy()
    total_runs = max(1, int(df["n_cv_runs"].max()))
    # better denominator = total ok folds across all combos if available
    if "n_total_ok_folds" in df.columns:
        total_runs = max(1, int(df["n_total_ok_folds"].iloc[0]))
    df["support_share"] = df["n_cv_runs"] / total_runs
    df["selection_objective"] = (
        df["mean_outer_r2"].astype(float)
        - SUMMARY_STD_PENALTY * df["std_outer_r2"].fillna(0.0).astype(float)
        + SUMMARY_SUPPORT_BONUS * df["support_share"].astype(float)
        - TOPK_PENALTY * df["topk"].astype(float)
    )
    min_share = float(MIN_SUPPORT_SHARE_BY_FAMILY.get(fam, 0.15))
    eligible = df[df["support_share"] >= min_share].copy()
    if eligible.empty:
        eligible = df.copy()
    eligible = eligible.sort_values(
        ["selection_objective", "mean_outer_r2", "std_outer_r2", "topk"],
        ascending=[False, False, True, True]
    ).reset_index(drop=True)
    return eligible.iloc[0]

def safe_signed_log1p_series(s):
    s = pd.to_numeric(pd.Series(s), errors="coerce")
    return np.sign(s) * np.log1p(np.abs(s))

def build_engineered_feature_space(X_df, base_ranked_cols, output_name, max_added=ENGINEERED_MAX_TOTAL_ADDED):
    X_df = X_df.copy()
    fam = classify_output_family(output_name)
    if (not ENGINEERED_FEATURES_ENABLED) or len(base_ranked_cols) == 0:
        return X_df, []

    base_cols = [c for c in base_ranked_cols if c in X_df.columns]
    base_cols = base_cols[:ENGINEERED_MAX_BASE_BY_FAMILY.get(fam, 4)]
    added_cols = []

    # signed log
    if ENGINEERED_INCLUDE_SIGNED_LOG:
        for c in base_cols:
            new_c = f"{c}__slog"
            if new_c not in X_df.columns:
                X_df[new_c] = safe_signed_log1p_series(X_df[c])
                added_cols.append(new_c)
            if len(added_cols) >= max_added:
                return X_df, added_cols

    # square
    if ENGINEERED_INCLUDE_SQUARE:
        for c in base_cols:
            new_c = f"{c}__sq"
            if new_c not in X_df.columns:
                x = pd.to_numeric(X_df[c], errors="coerce")
                X_df[new_c] = x * x
                added_cols.append(new_c)
            if len(added_cols) >= max_added:
                return X_df, added_cols

    # pairwise products
    if ENGINEERED_INCLUDE_PRODUCT_BY_FAMILY.get(fam, False):
        for i in range(len(base_cols)):
            for j in range(i + 1, len(base_cols)):
                a, b = base_cols[i], base_cols[j]
                new_c = f"{a}__mul__{b}"
                if new_c not in X_df.columns:
                    xa = pd.to_numeric(X_df[a], errors="coerce")
                    xb = pd.to_numeric(X_df[b], errors="coerce")
                    X_df[new_c] = xa * xb
                    added_cols.append(new_c)
                if len(added_cols) >= max_added:
                    return X_df, added_cols

    # pairwise ratios
    if ENGINEERED_INCLUDE_RATIO_BY_FAMILY.get(fam, False):
        for i in range(len(base_cols)):
            for j in range(len(base_cols)):
                if i == j:
                    continue
                a, b = base_cols[i], base_cols[j]
                new_c = f"{a}__div__{b}"
                if new_c not in X_df.columns:
                    xa = pd.to_numeric(X_df[a], errors="coerce")
                    xb = pd.to_numeric(X_df[b], errors="coerce")
                    denom = xb.where(xb.abs() > 1e-12, np.nan)
                    X_df[new_c] = xa / denom
                    added_cols.append(new_c)
                if len(added_cols) >= max_added:
                    return X_df, added_cols

    return X_df, added_cols


def prefilter_features_groupwise(X_df, y, groups, top_k=20, corr_threshold=0.90, random_state=42):
    Xg, yg, gg = aggregate_by_group_firstX(X_df, y, groups, y_method="median")
    X_num = Xg.copy().replace([np.inf, -np.inf], np.nan)
    valid_cols = [c for c in X_num.columns if X_num[c].notna().sum() >= 5]
    X_num = X_num[valid_cols].copy()

    spearman_scores = {}
    pearson_scores = {}
    mi_scores = {}
    enet_scores = {}

    for c in X_num.columns:
        tmp = pd.concat([X_num[[c]], yg], axis=1).dropna()
        if len(tmp) < 5:
            spearman_scores[c] = 0.0
            pearson_scores[c] = 0.0
            continue
        rho = spearmanr(tmp[c], tmp["target"]).statistic
        spearman_scores[c] = 0.0 if pd.isna(rho) else abs(float(rho))
        try:
            pr = np.corrcoef(tmp[c].astype(float), tmp["target"].astype(float))[0, 1]
            pearson_scores[c] = 0.0 if pd.isna(pr) else abs(float(pr))
        except Exception:
            pearson_scores[c] = 0.0

    X_imp = X_num.fillna(X_num.median(numeric_only=True))
    if X_imp.shape[1] > 0 and len(yg) >= 8:
        try:
            mi = mutual_info_regression(X_imp, yg, random_state=random_state)
            mi_scores = dict(zip(X_imp.columns, mi))
        except Exception:
            mi_scores = {c: 0.0 for c in X_imp.columns}
    else:
        mi_scores = {c: 0.0 for c in X_imp.columns}

    if X_imp.shape[1] > 0 and len(yg) >= 8:
        try:
            enet = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", ElasticNetCV(
                    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                    alphas=np.logspace(-4, 1, 40),
                    cv=min(5, len(yg)),
                    random_state=random_state,
                    max_iter=30000
                ))
            ])
            enet.fit(X_imp, yg)
            coef = np.abs(enet.named_steps["model"].coef_)
            enet_scores = dict(zip(X_imp.columns, coef))
        except Exception:
            enet_scores = {c: 0.0 for c in X_imp.columns}
    else:
        enet_scores = {c: 0.0 for c in X_imp.columns}

    score_df = pd.DataFrame({"feature_name": X_num.columns})
    score_df["spearman"] = score_df["feature_name"].map(spearman_scores).fillna(0.0)
    score_df["pearson"] = score_df["feature_name"].map(pearson_scores).fillna(0.0)
    score_df["mutual_info"] = score_df["feature_name"].map(mi_scores).fillna(0.0)
    score_df["elastic_net"] = score_df["feature_name"].map(enet_scores).fillna(0.0)

    for col in ["spearman", "pearson", "mutual_info", "elastic_net"]:
        score_df[col + "_norm"] = minmax_series(score_df[col])

    score_df["ensemble_score"] = (
        FEATURE_SCORE_WEIGHTS["spearman"] * score_df["spearman_norm"]
        + FEATURE_SCORE_WEIGHTS["pearson"] * score_df["pearson_norm"]
        + FEATURE_SCORE_WEIGHTS["mutual_info"] * score_df["mutual_info_norm"]
        + FEATURE_SCORE_WEIGHTS["elastic_net"] * score_df["elastic_net_norm"]
    )
    score_df = score_df.sort_values("ensemble_score", ascending=False).reset_index(drop=True)

    ranked = score_df["feature_name"].tolist()
    ranked = corr_prune_from_ranked(X_imp, ranked, keep_k=top_k, threshold=corr_threshold)

    return ranked, score_df


def make_target_transformer(method):
    if method == "raw":
        return None
    if method == "yeo_johnson":
        return PowerTransformer(method="yeo-johnson", standardize=True)
    return None

def make_pipeline_and_space(model_name, n_features, random_state=42):
    max_comp = max(1, min(n_features, 8))
    if model_name == "Bayesian_Ridge":
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", BayesianRidge())
        ])
        return pipe, {}, "grid", 1

    if model_name == "Ridge":
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(random_state=random_state))
        ])
        return pipe, {"model__alpha": loguniform(1e-4, 1e3)}, "random", 20

    if model_name == "ElasticNet_CV":
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNet(max_iter=30000, random_state=random_state))
        ])
        return pipe, {"model__alpha": loguniform(1e-4, 1e1), "model__l1_ratio": uniform(0.05, 0.90)}, "random", 24

    if model_name == "Huber_Regressor":
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", HuberRegressor(max_iter=3000))
        ])
        return pipe, {"model__alpha": loguniform(1e-6, 1e-1), "model__epsilon": uniform(1.15, 0.85)}, "random", 18

    if model_name == "PLS_Regression":
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PLSRegression())
        ])
        return pipe, {"model__n_components": list(range(1, max_comp + 1))}, "grid", 1

    if model_name == "PCR_Ridge":
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("pca", PCA()),
            ("model", Ridge(random_state=random_state))
        ])
        return pipe, {
            "pca__n_components": list(range(1, max_comp + 1)),
            "model__alpha": loguniform(1e-4, 1e3),
        }, "random", 24

    if model_name == "Kernel_Ridge_RBF":
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", KernelRidge(kernel="rbf"))
        ])
        return pipe, {
            "model__alpha": loguniform(1e-4, 1e2),
            "model__gamma": loguniform(1e-4, 1e1),
        }, "random", 24

    if model_name == "SVR_RBF":
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVR(kernel="rbf"))
        ])
        return pipe, {
            "model__C": loguniform(1e-2, 1e3),
            "model__gamma": loguniform(1e-4, 1e1),
            "model__epsilon": loguniform(1e-4, 5e-1),
        }, "random", 26

    raise ValueError(f"Unknown model: {model_name}")

def fit_search_model(X_train, y_train, groups_train, model_name, target_transform="raw", scoring_name="r2", random_state=42):
    pipe, params, search_type, n_iter = make_pipeline_and_space(model_name, X_train.shape[1], random_state=random_state)
    transformer = make_target_transformer(target_transform)
    estimator = pipe if transformer is None else TransformedTargetRegressor(
        regressor=pipe,
        transformer=transformer,
        check_inverse=False
    )

    wrapped_params = params if transformer is None else {f"regressor__{k}": v for k, v in params.items()}
    n_unique_groups = len(pd.unique(groups_train))
    if n_unique_groups < 4:
        cv = GroupKFold(n_splits=max(2, n_unique_groups))
    else:
        cv = GroupShuffleSplit(n_splits=INNER_GSS_SPLITS, test_size=INNER_GSS_TEST_SIZE, random_state=random_state)

    if search_type == "grid":
        search = GridSearchCV(
            estimator=estimator,
            param_grid=wrapped_params,
            scoring=scoring_name,
            cv=cv,
            n_jobs=SEARCH_N_JOBS,
            refit=True,
            return_train_score=False
        )
    else:
        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=wrapped_params,
            n_iter=n_iter,
            scoring=scoring_name,
            cv=cv,
            n_jobs=SEARCH_N_JOBS,
            refit=True,
            random_state=random_state,
            return_train_score=False
        )

    search.fit(X_train, y_train, groups=groups_train)
    train_pred = np.asarray(search.best_estimator_.predict(X_train)).reshape(-1)
    train_r2 = float(r2_score(y_train, train_pred)) if len(y_train) >= 2 else np.nan
    best_idx = int(search.best_index_)
    cv_res = pd.DataFrame(search.cv_results_)
    best_score_std = float(cv_res.loc[best_idx, "std_test_score"]) if "std_test_score" in cv_res.columns else np.nan
    return {
        "best_estimator": search.best_estimator_,
        "best_params": search.best_params_,
        "best_score": float(search.best_score_),
        "best_score_std": best_score_std,
        "train_r2": train_r2,
    }

def evaluate_fixed_estimator_groupcv(estimator, X, y, groups, scoring_name="r2", n_splits=5):
    X = pd.DataFrame(X).reset_index(drop=True)
    y = np.asarray(y, dtype=float)
    groups = np.asarray(groups)
    n_unique_groups = len(pd.unique(groups))
    n_splits = max(2, min(n_splits, n_unique_groups))
    cv = GroupKFold(n_splits=n_splits)
    fold_scores = []
    for tr_idx, va_idx in cv.split(X, y, groups):
        est = clone(estimator)
        est.fit(X.iloc[tr_idx], y[tr_idx])
        pred = np.asarray(est.predict(X.iloc[va_idx])).reshape(-1)
        if scoring_name == "neg_root_mean_squared_error":
            score = -float(math.sqrt(mean_squared_error(y[va_idx], pred)))
        elif scoring_name == "neg_mean_absolute_error":
            score = -float(mean_absolute_error(y[va_idx], pred))
        else:
            score = float(r2_score(y[va_idx], pred))
        fold_scores.append(score)
    if len(fold_scores) == 0:
        return np.nan, np.nan
    return float(np.mean(fold_scores)), float(np.std(fold_scores, ddof=0))

class WeightedBlendRegressor:
    def __init__(self, estimators, weights, model_names=None):
        self.estimators = estimators
        self.weights = np.asarray(weights, dtype=float)
        s = self.weights.sum()
        self.weights = self.weights / s if s > 0 else np.ones(len(self.weights)) / max(1, len(self.weights))
        self.model_names = model_names if model_names is not None else [f"base_{i+1}" for i in range(len(self.estimators))]

    def fit(self, X, y):
        self.fitted_estimators_ = []
        for est in self.estimators:
            e = clone(est)
            e.fit(X, y)
            self.fitted_estimators_.append(e)
        return self

    def predict(self, X):
        preds = []
        for est in self.fitted_estimators_:
            preds.append(np.asarray(est.predict(X)).reshape(-1))
        P = np.vstack(preds)
        return np.average(P, axis=0, weights=self.weights)

def fit_weighted_blend_estimators(base_estimators, weights, X_train, y_train, model_names=None):
    blend = WeightedBlendRegressor(base_estimators, weights, model_names=model_names)
    blend.fit(X_train, y_train)
    return blend

def evaluate_weighted_blend_groupcv(base_estimators, weights, X, y, groups, scoring_name="r2", n_splits=5):
    X = pd.DataFrame(X).reset_index(drop=True)
    y = np.asarray(y, dtype=float)
    groups = np.asarray(groups)
    n_unique_groups = len(pd.unique(groups))
    n_splits = max(2, min(n_splits, n_unique_groups))
    cv = GroupKFold(n_splits=n_splits)
    fold_scores = []
    for tr_idx, va_idx in cv.split(X, y, groups):
        preds = []
        for est in base_estimators:
            e = clone(est)
            e.fit(X.iloc[tr_idx], y[tr_idx])
            preds.append(np.asarray(e.predict(X.iloc[va_idx])).reshape(-1))
        P = np.vstack(preds)
        blend_pred = np.average(P, axis=0, weights=weights)
        if scoring_name == "neg_root_mean_squared_error":
            score = -float(math.sqrt(mean_squared_error(y[va_idx], blend_pred)))
        elif scoring_name == "neg_mean_absolute_error":
            score = -float(mean_absolute_error(y[va_idx], blend_pred))
        else:
            score = float(r2_score(y[va_idx], blend_pred))
        fold_scores.append(score)
    if len(fold_scores) == 0:
        return np.nan, np.nan
    return float(np.mean(fold_scores)), float(np.std(fold_scores, ddof=0))


def get_model_aware_oof_predictions(X_raw, y_raw, groups_raw, rough_cols, scoring_models=None, random_state=42):
    if scoring_models is None:
        scoring_models = MODEL_AWARE_SCORING_MODELS

    X_raw = X_raw.reset_index(drop=True)
    y_raw = pd.Series(np.asarray(y_raw, dtype=float), name="target").reset_index(drop=True)
    groups_raw = pd.Series(groups_raw).reset_index(drop=True)

    pred_lists = defaultdict(list)
    splits = list(repeated_group_splits(groups_raw.to_numpy(), n_splits=min(5, len(pd.unique(groups_raw))), n_repeats=8, random_state=random_state))

    for split_id, (tr_idx, va_idx) in enumerate(splits, start=1):
        X_tr_raw = X_raw.iloc[tr_idx].reset_index(drop=True)
        y_tr_raw = y_raw.iloc[tr_idx].reset_index(drop=True)
        g_tr_raw = groups_raw.iloc[tr_idx].reset_index(drop=True)

        X_va_raw = X_raw.iloc[va_idx].reset_index(drop=True)

        X_tr_g, y_tr_g, g_tr_g = aggregate_by_group_firstX(X_tr_raw[rough_cols], y_tr_raw.to_numpy(), g_tr_raw.to_numpy(), y_method="median")
        X_va = X_va_raw[rough_cols].copy()

        for model_name in scoring_models:
            try:
                if model_name == "PLS_Regression" and X_tr_g.shape[1] < 1:
                    continue
                fit_info = fit_search_model(X_tr_g, y_tr_g.to_numpy(), g_tr_g.to_numpy(), model_name=model_name, target_transform="raw", scoring_name="r2", random_state=random_state + split_id)
                pred_va = np.asarray(fit_info["best_estimator"].predict(X_va)).reshape(-1)
                for idx_global, pred in zip(va_idx, pred_va):
                    pred_lists[int(idx_global)].append(float(pred))
            except Exception:
                continue

    oof_pred = []
    fallback = float(np.nanmedian(y_raw))
    for i in range(len(y_raw)):
        vals = pred_lists.get(i, [])
        oof_pred.append(float(np.mean(vals)) if len(vals) else fallback)

    oof_pred = pd.Series(oof_pred, name="model_aware_oof_pred")
    return oof_pred

def select_best_y_within_group(X_raw, y_raw, groups_raw, xkey_raw, rough_cols, strategy="hard_closest_oof", random_state=42):
    X_raw = X_raw.reset_index(drop=True)
    y_raw = pd.Series(np.asarray(y_raw, dtype=float), name="target").reset_index(drop=True)
    groups_raw = pd.Series(groups_raw, name="GROUP_ID").reset_index(drop=True)
    xkey_raw = pd.Series(xkey_raw, name="X_KEY").reset_index(drop=True)

    tmp = pd.concat([X_raw.copy(), y_raw, groups_raw, xkey_raw], axis=1)

    if strategy == "hard_closest_oof":
        oof_pred = get_model_aware_oof_predictions(X_raw, y_raw, groups_raw, rough_cols=rough_cols, random_state=random_state)
        tmp = pd.concat([tmp, oof_pred], axis=1)
        tmp["abs_resid_to_oof"] = (tmp["target"] - tmp["model_aware_oof_pred"]).abs()
        tmp["group_median_y"] = tmp.groupby("GROUP_ID")["target"].transform("median")
        tmp["abs_to_group_median"] = (tmp["target"] - tmp["group_median_y"]).abs()

        selected_rows = []
        selected_index = []
        for g, block in tmp.groupby("GROUP_ID", sort=False):
            block = block.sort_values(["abs_resid_to_oof", "abs_to_group_median"]).copy()
            chosen = block.iloc[[0]].copy()
            selected_rows.append(chosen)
            selected_index.append(chosen.index[0])

        selected_df = pd.concat(selected_rows, axis=0).reset_index(drop=True)
        tmp["selected_as_final_y"] = 0
        tmp.loc[selected_index, "selected_as_final_y"] = 1
        tmp["selected_strategy"] = strategy
        return selected_df, tmp

    # robust strategy: create one representative target per group
    selected_rows = []
    candidate_rows = []
    for g, block in tmp.groupby("GROUP_ID", sort=False):
        rep_y = robust_group_target(block["target"].to_numpy(), z=ROBUST_TRIM_Z)
        chosen = block.iloc[[0]].copy()
        chosen["target"] = rep_y
        chosen["model_aware_oof_pred"] = np.nan
        chosen["abs_resid_to_oof"] = np.nan
        chosen["group_median_y"] = float(pd.Series(block["target"]).median())
        chosen["abs_to_group_median"] = abs(rep_y - chosen["group_median_y"].iloc[0])
        chosen["selected_as_final_y"] = 1
        chosen["selected_strategy"] = strategy
        selected_rows.append(chosen)

        b = block.copy()
        b["model_aware_oof_pred"] = np.nan
        b["abs_resid_to_oof"] = np.nan
        b["group_median_y"] = float(pd.Series(block["target"]).median())
        b["abs_to_group_median"] = (b["target"] - b["group_median_y"]).abs()
        b["selected_as_final_y"] = 0
        b["selected_strategy"] = strategy
        b["group_representative_y"] = rep_y
        candidate_rows.append(b)

    selected_df = pd.concat(selected_rows, axis=0).reset_index(drop=True)
    tmp = pd.concat(candidate_rows, axis=0).reset_index(drop=True)
    return selected_df, tmp

def aggregate_test_groups_by_strategy(X_raw, y_raw, groups_raw, xkey_raw, strategy="hard_closest_oof"):
    X_raw = X_raw.reset_index(drop=True)
    y_raw = pd.Series(np.asarray(y_raw, dtype=float), name="target").reset_index(drop=True)
    groups_raw = pd.Series(groups_raw, name="GROUP_ID").reset_index(drop=True)
    xkey_raw = pd.Series(xkey_raw, name="X_KEY").reset_index(drop=True)
    tmp = pd.concat([X_raw.copy(), y_raw, groups_raw, xkey_raw], axis=1)

    rows = []
    for g, block in tmp.groupby("GROUP_ID", sort=False):
        row = block.iloc[[0]].copy()
        if strategy == "robust_trimmed_median":
            row["target"] = robust_group_target(block["target"].to_numpy(), z=ROBUST_TRIM_Z)
        else:
            row["target"] = float(pd.Series(block["target"]).median())
        rows.append(row)
    out = pd.concat(rows, axis=0).reset_index(drop=True)
    return out

def build_feature_frequency(X_df, y, groups, candidate_cols, n_repeats=24, random_state=42):
    candidate_cols = list(candidate_cols)
    freq = pd.Series(0.0, index=candidate_cols, dtype=float)
    if len(candidate_cols) == 0:
        return pd.DataFrame({"feature_name": [], "selection_frequency": []})

    splits = list(repeated_group_splits(groups, n_splits=min(5, len(pd.unique(groups))), n_repeats=n_repeats, random_state=random_state))
    if len(splits) == 0:
        return pd.DataFrame({"feature_name": candidate_cols, "selection_frequency": np.zeros(len(candidate_cols))})

    for split_id, (tr_idx, va_idx) in enumerate(splits, start=1):
        X_tr = X_df.iloc[tr_idx][candidate_cols].copy()
        y_tr = np.asarray(y)[tr_idx]
        if len(np.unique(y_tr)) < 2:
            continue
        try:
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", ElasticNetCV(
                    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                    alphas=np.logspace(-4, 1, 40),
                    cv=min(5, len(y_tr)),
                    random_state=random_state + split_id,
                    max_iter=30000
                ))
            ])
            pipe.fit(X_tr, y_tr)
            coef = np.abs(pipe.named_steps["model"].coef_)
            selected = [f for f, c in zip(candidate_cols, coef) if abs(c) > 1e-12]
            for f in selected:
                freq[f] += 1.0
        except Exception:
            continue

    if freq.max() > 0:
        freq = freq / freq.max()
    out = freq.sort_values(ascending=False).rename("selection_frequency").reset_index()
    out.columns = ["feature_name", "selection_frequency"]
    return out


def get_topk_candidates_for_output(output_name):
    fam = classify_output_family(output_name)
    return TOPK_BY_FAMILY.get(fam, FINAL_TOPK_CANDIDATES)

def get_best_y_strategy_for_output(output_name, groups=None):
    fam = classify_output_family(output_name)
    if APPLY_REPEATED_Y_ONLY_IF_NEEDED and groups is not None:
        rep = group_repeat_stats(groups)
        if not rep["has_repeats"]:
            return "hard_closest_oof" if fam == "mechanical_energy" else "robust_trimmed_median"
    if fam in ("thermal", "vibrational"):
        return "robust_trimmed_median"
    return "hard_closest_oof"


def robust_group_target(block_y, z=ROBUST_TRIM_Z):
    y = pd.Series(np.asarray(block_y, dtype=float)).dropna()
    if len(y) == 0:
        return np.nan
    if len(y) <= 2:
        return float(y.median())
    med = float(y.median())
    mad = float(np.median(np.abs(y - med)))
    if mad < 1e-12:
        return float(y.median())
    robust_z = 0.6745 * (y - med).abs() / mad
    keep = y[robust_z <= z]
    if len(keep) == 0:
        keep = y
    return float(keep.median())

def get_candidate_models_for_output(output_name):
    fam = classify_output_family(output_name)
    if fam == "vibrational":
        return ["PCR_Ridge", "PLS_Regression", "Bayesian_Ridge", "Huber_Regressor", "Ridge", "Kernel_Ridge_RBF", "SVR_RBF"]
    if fam == "thermal":
        return ["Huber_Regressor", "Bayesian_Ridge", "PCR_Ridge", "PLS_Regression", "Ridge", "ElasticNet_CV", "Kernel_Ridge_RBF", "SVR_RBF"]
    return ["PCR_Ridge", "PLS_Regression", "Bayesian_Ridge", "Ridge", "Huber_Regressor", "ElasticNet_CV"]

def reorder_ranked_with_consensus(ranked_features, consensus_features):
    consensus_features = [f for f in consensus_features if f in ranked_features]
    return consensus_features + [f for f in ranked_features if f not in consensus_features]

def choose_best_model_and_topk(X_train, y_train, g_train, ranked_features, output_name="", target_transform_candidates=None, model_names=None, random_state=42):
    if target_transform_candidates is None:
        target_transform_candidates = TARGET_TRANSFORM_CANDIDATES
    if model_names is None:
        model_names = FINAL_MODEL_NAMES

    rows = []
    fam = classify_output_family(output_name)
    scoring_name = get_inner_scoring_for_output(output_name)

    for topk in get_topk_candidates_for_output(output_name):
        feat_list = ranked_features[:min(topk, len(ranked_features))]
        if len(feat_list) < 2:
            continue
        X_sel = X_train[feat_list].copy()

        for tt in target_transform_candidates:
            group_candidates = []
            for model_name in model_names:
                try:
                    fit_info = fit_search_model(
                        X_sel, y_train, g_train,
                        model_name=model_name,
                        target_transform=tt,
                        scoring_name=scoring_name,
                        random_state=random_state
                    )
                    score_std = float(fit_info.get("best_score_std", np.nan))
                    stability_objective = (
                        float(fit_info["best_score"])
                        - 0.08 * (0.0 if pd.isna(score_std) else score_std)
                        - TOPK_PENALTY * float(topk)
                        - COMPLEXITY_PENALTY * float(MODEL_COMPLEXITY_RANK[model_name])
                    )
                    row = {
                        "model_name": model_name,
                        "target_transform": tt,
                        "topk": int(topk),
                        "selected_features": feat_list,
                        "best_estimator": fit_info["best_estimator"],
                        "best_params": fit_info["best_params"],
                        "search_score": float(fit_info["best_score"]),
                        "search_score_std": score_std,
                        "search_scoring": scoring_name,
                        "stability_objective": float(stability_objective),
                        "train_r2": float(fit_info.get("train_r2", np.nan)),
                        "complexity_rank": int(MODEL_COMPLEXITY_RANK[model_name]),
                    }
                    rows.append(row)
                    group_candidates.append(row)
                except Exception:
                    continue

            # weighted blend over top-performing base models for same topk/transform
            if len(group_candidates) >= 2:
                tmp_df = pd.DataFrame(group_candidates).sort_values(
                    ["search_score", "search_score_std", "complexity_rank"],
                    ascending=[False, True, True]
                ).reset_index(drop=True)

                for blend_size in BLEND_TOP_CANDIDATES:
                    if len(tmp_df) < blend_size:
                        continue
                    top_df = tmp_df.head(blend_size).copy()
                    raw_scores = top_df["search_score"].astype(float).to_numpy()
                    keep_mask = raw_scores >= BLEND_MIN_BASE_SCORE
                    if keep_mask.sum() < 2:
                        continue
                    top_df = top_df.loc[keep_mask].reset_index(drop=True)
                    raw_scores = top_df["search_score"].astype(float).to_numpy()
                    weights = np.maximum(raw_scores - raw_scores.min() + 1e-6, 1e-6)
                    weights = weights / weights.sum()

                    try:
                        blend_cv_mean, blend_cv_std = evaluate_weighted_blend_groupcv(
                            base_estimators=top_df["best_estimator"].tolist(),
                            weights=weights,
                            X=X_sel,
                            y=y_train,
                            groups=g_train,
                            scoring_name=scoring_name,
                            n_splits=BLEND_EVAL_FOLDS,
                        )
                        blend_estimator = fit_weighted_blend_estimators(
                            base_estimators=top_df["best_estimator"].tolist(),
                            weights=weights,
                            X_train=X_sel,
                            y_train=y_train,
                            model_names=top_df["model_name"].tolist(),
                        )
                        blend_name = f"WeightedBlend_{len(top_df)}"
                        stability_objective = (
                            float(blend_cv_mean)
                            - 0.08 * (0.0 if pd.isna(blend_cv_std) else blend_cv_std)
                            - TOPK_PENALTY * float(topk)
                            - COMPLEXITY_PENALTY * float(MODEL_COMPLEXITY_RANK[blend_name])
                        )
                        rows.append({
                            "model_name": blend_name,
                            "target_transform": tt,
                            "topk": int(topk),
                            "selected_features": feat_list,
                            "best_estimator": blend_estimator,
                            "best_params": {
                                "blend_models": top_df["model_name"].tolist(),
                                "blend_weights": [float(w) for w in weights],
                            },
                            "search_score": float(blend_cv_mean),
                            "search_score_std": float(blend_cv_std),
                            "search_scoring": scoring_name,
                            "stability_objective": float(stability_objective),
                            "train_r2": float(np.nan),
                            "complexity_rank": int(MODEL_COMPLEXITY_RANK[blend_name]),
                        })
                    except Exception:
                        pass

    df = pd.DataFrame(rows)
    if df.empty:
        return None, df

    best = df.sort_values(
        ["stability_objective", "search_score", "search_score_std", "topk", "complexity_rank"],
        ascending=[False, False, True, True, True]
    ).iloc[0].to_dict()
    return best, df


In [7]:
# ============================================================
# Cell D1. Repeated outer-CV evaluation for each output
#  - NaN-safe version
# ============================================================

ALL_SUMMARY_ROWS = []
ALL_FOLD_ROWS = []
ALL_MODEL_SEARCH_ROWS = []
PER_OUTPUT_RESULTS = {}

for output_name, bundle in DATA_BY_OUTPUT.items():
    df = bundle["df"].copy()
    feature_cols = bundle["feature_cols"]
    target_col = bundle["target_col"]
    group_col = bundle["group_col"]
    xkey_col = bundle["xkey_col"]

    # ---------- full raw ----------
    X_raw_full = df[feature_cols].copy()
    y_raw_full = pd.to_numeric(df[target_col], errors="coerce").to_numpy(dtype=float)
    g_raw_full = df[group_col].to_numpy()
    xkey_full = df[xkey_col].to_numpy()

    # global valid mask: target/group/xkey missing rows remove first
    base_mask = (
        np.isfinite(y_raw_full)
        & pd.notna(g_raw_full)
        & pd.notna(xkey_full)
    )
    X_raw_full = X_raw_full.loc[base_mask].reset_index(drop=True)
    y_raw_full = y_raw_full[base_mask]
    g_raw_full = g_raw_full[base_mask]
    xkey_full = xkey_full[base_mask]

    n_groups = len(pd.unique(g_raw_full))
    if n_groups < MIN_GROUPS_FOR_MODEL:
        ALL_SUMMARY_ROWS.append({
            "output_name": output_name,
            "status": "skipped",
            "reason": f"Not enough groups ({n_groups})",
        })
        continue

    best_y_strategy = get_best_y_strategy_for_output(output_name, groups=g_raw_full)
    fold_rows = []
    fold_model_rows = []

    split_iter = repeated_group_splits(
        g_raw_full,
        n_splits=min(FINAL_OUTER_SPLITS, n_groups),
        n_repeats=OUTER_REPEATS,
        random_state=RANDOM_STATE,
        return_split_id=True,
    )

    for cv_run_id, repeat_id, fold_id, tr_idx, te_idx in split_iter:
        try:
            # ---------- split ----------
            X_tr_raw = X_raw_full.iloc[tr_idx].reset_index(drop=True)
            y_tr_raw = y_raw_full[tr_idx]
            g_tr_raw = g_raw_full[tr_idx]
            xkey_tr = xkey_full[tr_idx]

            X_te_raw = X_raw_full.iloc[te_idx].reset_index(drop=True)
            y_te_raw = y_raw_full[te_idx]
            g_te_raw = g_raw_full[te_idx]
            xkey_te = xkey_full[te_idx]

            # ---------- rough prefilter ----------
            rough_cols, rough_score_df = prefilter_features_groupwise(
                X_tr_raw, y_tr_raw, g_tr_raw,
                top_k=get_prefilter_topk_for_output(output_name),
                corr_threshold=CORR_PRUNE_THRESHOLD,
                random_state=RANDOM_STATE + cv_run_id
            )
            if len(rough_cols) < 2:
                fold_rows.append({
                    "output_name": output_name,
                    "cv_run_id": int(cv_run_id),
                    "repeat_id": int(repeat_id),
                    "fold_id": int(fold_id),
                    "best_y_strategy": best_y_strategy,
                    "output_family": classify_output_family(output_name),
                    "model_name": None,
                    "target_transform": None,
                    "topk": np.nan,
                    "n_train_groups": int(len(pd.unique(g_tr_raw))),
                    "n_test_groups": int(len(pd.unique(g_te_raw))),
                    "n_features": 0,
                    "selected_features": "",
                    "outer_r2": np.nan,
                    "outer_rmse": np.nan,
                    "outer_mae": np.nan,
                    "search_score": np.nan,
                    "train_r2_inner": np.nan,
                    "candidate_model_pool": "",
                    "status": "skipped",
                    "reason": "rough_cols < 2",
                })
                continue

            # ---------- conservative feature engineering on rough-top features ----------
            X_tr_aug, added_tr_cols = build_engineered_feature_space(X_tr_raw, rough_cols, output_name=output_name)
            X_te_aug, added_te_cols = build_engineered_feature_space(X_te_raw, rough_cols, output_name=output_name)
            aug_feature_cols = list(X_tr_aug.columns)

            # ---------- repeated-Y selection / aggregation ----------
            selected_train_df, train_candidates_df = select_best_y_within_group(
                X_tr_aug, y_tr_raw, g_tr_raw, xkey_tr,
                rough_cols=rough_cols,
                strategy=best_y_strategy,
                random_state=RANDOM_STATE + cv_run_id
            )
            test_group_df = aggregate_test_groups_by_strategy(
                X_te_aug, y_te_raw, g_te_raw, xkey_te,
                strategy=best_y_strategy
            )

            # safety: essential columns
            selected_train_df = selected_train_df.copy()
            test_group_df = test_group_df.copy()

            # remove target/group missing
            selected_train_df["target"] = pd.to_numeric(selected_train_df["target"], errors="coerce")
            test_group_df["target"] = pd.to_numeric(test_group_df["target"], errors="coerce")

            selected_train_df = selected_train_df.dropna(subset=["target", "GROUP_ID"]).reset_index(drop=True)
            test_group_df = test_group_df.dropna(subset=["target", "GROUP_ID"]).reset_index(drop=True)

            if selected_train_df.empty or test_group_df.empty:
                fold_rows.append({
                    "output_name": output_name,
                    "cv_run_id": int(cv_run_id),
                    "repeat_id": int(repeat_id),
                    "fold_id": int(fold_id),
                    "best_y_strategy": best_y_strategy,
                    "output_family": classify_output_family(output_name),
                    "model_name": None,
                    "target_transform": None,
                    "topk": np.nan,
                    "n_train_groups": int(selected_train_df["GROUP_ID"].nunique()) if "GROUP_ID" in selected_train_df.columns else 0,
                    "n_test_groups": int(test_group_df["GROUP_ID"].nunique()) if "GROUP_ID" in test_group_df.columns else 0,
                    "n_features": 0,
                    "selected_features": "",
                    "outer_r2": np.nan,
                    "outer_rmse": np.nan,
                    "outer_mae": np.nan,
                    "search_score": np.nan,
                    "train_r2_inner": np.nan,
                    "candidate_model_pool": "",
                    "status": "skipped",
                    "reason": "selected_train_df or test_group_df empty after target/group cleanup",
                })
                continue

            X_train_sel = selected_train_df[aug_feature_cols].copy()
            y_train_sel = selected_train_df["target"].to_numpy(dtype=float)
            g_train_sel = selected_train_df["GROUP_ID"].to_numpy()

            X_test_group = test_group_df[aug_feature_cols].copy()
            y_test_group = test_group_df["target"].to_numpy(dtype=float)
            g_test_group = test_group_df["GROUP_ID"].to_numpy()

            # ---------- feature ranking ----------
            final_prefilter_cols, final_score_df = prefilter_features_groupwise(
                X_train_sel, y_train_sel, g_train_sel,
                top_k=get_prefilter_topk_for_output(output_name),
                corr_threshold=CORR_PRUNE_THRESHOLD,
                random_state=RANDOM_STATE + 1000 + cv_run_id
            )
            if len(final_prefilter_cols) < 2:
                fold_rows.append({
                    "output_name": output_name,
                    "cv_run_id": int(cv_run_id),
                    "repeat_id": int(repeat_id),
                    "fold_id": int(fold_id),
                    "best_y_strategy": best_y_strategy,
                    "output_family": classify_output_family(output_name),
                    "model_name": None,
                    "target_transform": None,
                    "topk": np.nan,
                    "n_train_groups": int(len(pd.unique(g_train_sel))),
                    "n_test_groups": int(len(pd.unique(g_test_group))),
                    "n_features": 0,
                    "selected_features": "",
                    "outer_r2": np.nan,
                    "outer_rmse": np.nan,
                    "outer_mae": np.nan,
                    "search_score": np.nan,
                    "train_r2_inner": np.nan,
                    "candidate_model_pool": "",
                    "status": "skipped",
                    "reason": "final_prefilter_cols < 2",
                })
                continue

            freq_df = build_feature_frequency(
                X_train_sel, y_train_sel, g_train_sel,
                candidate_cols=final_prefilter_cols,
                n_repeats=20,
                random_state=RANDOM_STATE + 2000 + cv_run_id
            )

            rank_df = (
                final_score_df[["feature_name", "ensemble_score"]]
                .merge(freq_df, on="feature_name", how="left")
                .fillna(0.0)
            )
            rank_df["rank_score"] = (
                0.55 * minmax_series(rank_df["selection_frequency"])
                + 0.45 * minmax_series(rank_df["ensemble_score"])
            )
            rank_df = rank_df.sort_values("rank_score", ascending=False).reset_index(drop=True)

            ranked_features = rank_df["feature_name"].tolist()
            ranked_features = corr_prune_from_ranked(
                X_train_sel[ranked_features], ranked_features,
                keep_k=MAX_FINAL_FEATURES,
                threshold=CORR_PRUNE_THRESHOLD
            )
            if len(ranked_features) < 2:
                fold_rows.append({
                    "output_name": output_name,
                    "cv_run_id": int(cv_run_id),
                    "repeat_id": int(repeat_id),
                    "fold_id": int(fold_id),
                    "best_y_strategy": best_y_strategy,
                    "output_family": classify_output_family(output_name),
                    "model_name": None,
                    "target_transform": None,
                    "topk": np.nan,
                    "n_train_groups": int(len(pd.unique(g_train_sel))),
                    "n_test_groups": int(len(pd.unique(g_test_group))),
                    "n_features": 0,
                    "selected_features": "",
                    "outer_r2": np.nan,
                    "outer_rmse": np.nan,
                    "outer_mae": np.nan,
                    "search_score": np.nan,
                    "train_r2_inner": np.nan,
                    "candidate_model_pool": "",
                    "status": "skipped",
                    "reason": "ranked_features < 2 after corr prune",
                })
                continue

            # ---------- inner model search ----------
            candidate_models = get_candidate_models_for_output(output_name)
            best_choice, model_search_df = choose_best_model_and_topk(
                X_train=X_train_sel[ranked_features],
                y_train=y_train_sel,
                g_train=g_train_sel,
                ranked_features=ranked_features,
                output_name=output_name,
                target_transform_candidates=TARGET_TRANSFORM_CANDIDATES,
                model_names=candidate_models,
                random_state=RANDOM_STATE + 3000 + cv_run_id
            )
            if best_choice is None:
                fold_rows.append({
                    "output_name": output_name,
                    "cv_run_id": int(cv_run_id),
                    "repeat_id": int(repeat_id),
                    "fold_id": int(fold_id),
                    "best_y_strategy": best_y_strategy,
                    "output_family": classify_output_family(output_name),
                    "model_name": None,
                    "target_transform": None,
                    "topk": np.nan,
                    "n_train_groups": int(len(pd.unique(g_train_sel))),
                    "n_test_groups": int(len(pd.unique(g_test_group))),
                    "n_features": 0,
                    "selected_features": "",
                    "outer_r2": np.nan,
                    "outer_rmse": np.nan,
                    "outer_mae": np.nan,
                    "search_score": np.nan,
                    "train_r2_inner": np.nan,
                    "candidate_model_pool": ", ".join(candidate_models),
                    "status": "skipped",
                    "reason": "best_choice is None",
                })
                continue

            # ---------- selected features ----------
            best_features = list(best_choice["selected_features"])
            best_estimator = best_choice["best_estimator"]

            # remove rows with NaN in selected feature set
            train_mask = np.isfinite(X_train_sel[best_features]).all(axis=1) & np.isfinite(y_train_sel)
            test_mask = np.isfinite(X_test_group[best_features]).all(axis=1) & np.isfinite(y_test_group)

            X_train_eval = X_train_sel.loc[train_mask, best_features].copy()
            y_train_eval = y_train_sel[train_mask]
            g_train_eval = g_train_sel[train_mask]

            X_test_eval = X_test_group.loc[test_mask, best_features].copy()
            y_test_eval = y_test_group[test_mask]
            g_test_eval = g_test_group[test_mask]

            if len(X_test_eval) < MIN_VALID_EVAL_SAMPLES:
                fold_rows.append({
                    "output_name": output_name,
                    "cv_run_id": int(cv_run_id),
                    "repeat_id": int(repeat_id),
                    "fold_id": int(fold_id),
                    "best_y_strategy": best_y_strategy,
                    "output_family": classify_output_family(output_name),
                    "model_name": best_choice["model_name"],
                    "target_transform": best_choice["target_transform"],
                    "topk": int(best_choice["topk"]),
                    "n_train_groups": int(len(pd.unique(g_train_eval))),
                    "n_test_groups": int(len(pd.unique(g_test_eval))),
                    "n_features": int(len(best_features)),
                    "selected_features": ", ".join(best_features),
                    "outer_r2": np.nan,
                    "outer_rmse": np.nan,
                    "outer_mae": np.nan,
                    "search_score": float(best_choice["search_score"]),
                    "train_r2_inner": float(best_choice.get("train_r2", np.nan)),
                    "candidate_model_pool": ", ".join(candidate_models),
                    "n_engineered_added": int(len([c for c in best_features if "__" in c])),
                    "status": "skipped",
                    "reason": "X_test_eval < MIN_VALID_EVAL_SAMPLES after NaN cleanup",
                })
                continue

            # predict
            pred_test = np.asarray(best_estimator.predict(X_test_eval)).reshape(-1)
            pred_test = pd.to_numeric(pd.Series(pred_test), errors="coerce").to_numpy(dtype=float)

            # final valid mask for metric computation
            valid_eval_mask = np.isfinite(y_test_eval) & np.isfinite(pred_test)

            if valid_eval_mask.sum() < MIN_VALID_EVAL_SAMPLES:
                fold_rows.append({
                    "output_name": output_name,
                    "cv_run_id": int(cv_run_id),
                    "repeat_id": int(repeat_id),
                    "fold_id": int(fold_id),
                    "best_y_strategy": best_y_strategy,
                    "output_family": classify_output_family(output_name),
                    "model_name": best_choice["model_name"],
                    "target_transform": best_choice["target_transform"],
                    "topk": int(best_choice["topk"]),
                    "n_train_groups": int(len(pd.unique(g_train_eval))),
                    "n_test_groups": int(len(pd.unique(g_test_eval))),
                    "n_features": int(len(best_features)),
                    "selected_features": ", ".join(best_features),
                    "outer_r2": np.nan,
                    "outer_rmse": np.nan,
                    "outer_mae": np.nan,
                    "search_score": float(best_choice["search_score"]),
                    "train_r2_inner": float(best_choice.get("train_r2", np.nan)),
                    "candidate_model_pool": ", ".join(candidate_models),
                    "status": "skipped",
                    "reason": "valid_eval_mask < MIN_VALID_EVAL_SAMPLES after prediction cleanup",
                })
                continue

            y_test_eval = y_test_eval[valid_eval_mask]
            pred_test = pred_test[valid_eval_mask]

            fold_rows.append({
                "output_name": output_name,
                "cv_run_id": int(cv_run_id),
                "repeat_id": int(repeat_id),
                "fold_id": int(fold_id),
                "best_y_strategy": best_y_strategy,
                "output_family": classify_output_family(output_name),
                "model_name": best_choice["model_name"],
                "target_transform": best_choice["target_transform"],
                "topk": int(best_choice["topk"]),
                "n_train_groups": int(len(pd.unique(g_train_eval))),
                "n_test_groups": int(len(pd.unique(g_test_eval))),
                "n_features": int(len(best_features)),
                "selected_features": ", ".join(best_features),
                "outer_r2": float(r2_score(y_test_eval, pred_test)),
                "outer_rmse": float(math.sqrt(mean_squared_error(y_test_eval, pred_test))),
                "outer_mae": float(mean_absolute_error(y_test_eval, pred_test)),
                "search_score": float(best_choice["search_score"]),
                "train_r2_inner": float(best_choice.get("train_r2", np.nan)),
                "candidate_model_pool": ", ".join(candidate_models),
                "status": "ok",
                "reason": "",
            })

            tmp_search = model_search_df.copy()
            tmp_search["output_name"] = output_name
            tmp_search["cv_run_id"] = cv_run_id
            tmp_search["repeat_id"] = repeat_id
            tmp_search["fold_id"] = fold_id
            tmp_search["best_y_strategy"] = best_y_strategy
            tmp_search["n_ranked_features"] = len(ranked_features)
            tmp_search["selected_features"] = tmp_search["selected_features"].apply(
                lambda x: ", ".join(x) if isinstance(x, (list, tuple, np.ndarray)) else str(x)
            )
            fold_model_rows.append(tmp_search)

        except Exception as e:
            fold_rows.append({
                "output_name": output_name,
                "cv_run_id": int(cv_run_id),
                "repeat_id": int(repeat_id),
                "fold_id": int(fold_id),
                "best_y_strategy": best_y_strategy,
                "output_family": classify_output_family(output_name),
                "model_name": None,
                "target_transform": None,
                "topk": np.nan,
                "n_train_groups": np.nan,
                "n_test_groups": np.nan,
                "n_features": 0,
                "selected_features": "",
                "outer_r2": np.nan,
                "outer_rmse": np.nan,
                "outer_mae": np.nan,
                "search_score": np.nan,
                "train_r2_inner": np.nan,
                "candidate_model_pool": "",
                "status": "error",
                "reason": f"{type(e).__name__}: {e}",
            })
            continue

    fold_df = pd.DataFrame(fold_rows)

    # keep only successful folds for summary
    ok_fold_df = fold_df.loc[fold_df["status"] == "ok"].copy()

    if ok_fold_df.empty:
        ALL_SUMMARY_ROWS.append({
            "output_name": output_name,
            "status": "failed",
            "reason": "No valid repeated outer-CV result",
        })

        out_dir = os.path.join(PER_OUTPUT_DIR, sanitize_filename(output_name))
        os.makedirs(out_dir, exist_ok=True)
        export_df(fold_df, os.path.join(out_dir, "outer_fold_metrics_repeated"))
        continue

    model_summary_df = (
        ok_fold_df.groupby(
            ["best_y_strategy", "model_name", "target_transform", "topk"],
            as_index=False
        )
        .agg(
            mean_outer_r2=("outer_r2", "mean"),
            std_outer_r2=("outer_r2", "std"),
            mean_outer_rmse=("outer_rmse", "mean"),
            mean_outer_mae=("outer_mae", "mean"),
            n_cv_runs=("cv_run_id", "count"),
        )
        .reset_index(drop=True)
    )
    model_summary_df["n_total_ok_folds"] = int(len(ok_fold_df))
    model_summary_df["support_share"] = model_summary_df["n_cv_runs"] / max(1, len(ok_fold_df))
    model_summary_df["selection_objective"] = (
        model_summary_df["mean_outer_r2"].astype(float)
        - SUMMARY_STD_PENALTY * model_summary_df["std_outer_r2"].fillna(0.0).astype(float)
        + SUMMARY_SUPPORT_BONUS * model_summary_df["support_share"].astype(float)
        - TOPK_PENALTY * model_summary_df["topk"].astype(float)
    )
    model_summary_df = model_summary_df.sort_values(
        ["selection_objective", "mean_outer_r2", "std_outer_r2", "support_share", "topk"],
        ascending=[False, False, True, False, True]
    ).reset_index(drop=True)

    best_row = choose_best_summary_row(model_summary_df, output_name)
    ALL_SUMMARY_ROWS.append({
        "output_name": output_name,
        "status": "ok",
        "reason": "",
        "output_family": classify_output_family(output_name),
        "best_y_strategy": best_row["best_y_strategy"],
        "final_model_name": best_row["model_name"],
        "target_transform": best_row["target_transform"],
        "final_chosen_topk": int(best_row["topk"]),
        "n_cv_runs": int(best_row["n_cv_runs"]),
        "cv_r2_mean": float(best_row["mean_outer_r2"]),
        "cv_r2_std": float(0.0 if pd.isna(best_row["std_outer_r2"]) else best_row["std_outer_r2"]),
        "cv_rmse_mean": float(best_row["mean_outer_rmse"]),
        "cv_mae_mean": float(best_row["mean_outer_mae"]),
        "candidate_model_pool": ", ".join(get_candidate_models_for_output(output_name)),
    })

    out_dir = os.path.join(PER_OUTPUT_DIR, sanitize_filename(output_name))
    os.makedirs(out_dir, exist_ok=True)

    export_df(fold_df, os.path.join(out_dir, "outer_fold_metrics_repeated"))
    export_df(ok_fold_df, os.path.join(out_dir, "outer_fold_metrics_repeated_ok_only"))
    export_df(model_summary_df, os.path.join(out_dir, "outer_model_summary_repeated"))

    if len(fold_model_rows):
        fold_model_df = pd.concat(fold_model_rows, axis=0).reset_index(drop=True)
        export_df(fold_model_df, os.path.join(out_dir, "outer_fold_model_search_repeated"))
        ALL_MODEL_SEARCH_ROWS.extend(fold_model_df.to_dict(orient="records"))

    ALL_FOLD_ROWS.extend(fold_df.to_dict(orient="records"))
    PER_OUTPUT_RESULTS[output_name] = {
        "fold_df": fold_df,
        "ok_fold_df": ok_fold_df,
        "model_summary_df": model_summary_df,
    }

ALL_SUMMARY_DF = pd.DataFrame(ALL_SUMMARY_ROWS)
ALL_FOLD_DF = pd.DataFrame(ALL_FOLD_ROWS)
ALL_MODEL_SEARCH_DF = pd.DataFrame(ALL_MODEL_SEARCH_ROWS)

export_df(ALL_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_summary"))
export_df(ALL_FOLD_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_fold_metrics"))
if not ALL_MODEL_SEARCH_DF.empty:
    export_df(ALL_MODEL_SEARCH_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_model_search"))

display(ALL_SUMMARY_DF.sort_values("cv_r2_mean", ascending=False))

,output_name,status,reason,output_family,best_y_strategy,final_model_name,target_transform,final_chosen_topk,n_cv_runs,cv_r2_mean,cv_r2_std,cv_rmse_mean,cv_mae_mean,candidate_model_pool
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,ok,,vibrational,robust_trimmed_median,WeightedBlend_2,yeo_johnson,2,1,0.967121,0.000000,0.016465,0.011178,"PCR_Ridge, PLS_Regression, Bayesian_Ridge, Hub..."
14,Vibrational response | FRF (g/N) | 3000-6500 H...,ok,,vibrational,robust_trimmed_median,WeightedBlend_3,raw,4,1,0.888945,0.000000,0.075902,0.053278,"PCR_Ridge, PLS_Regression, Bayesian_Ridge, Hub..."
12,Vibrational response | FRF (g/N) | 300-8000 Hz...,ok,,vibrational,robust_trimmed_median,WeightedBlend_3,yeo_johnson,4,2,0.778936,0.010679,0.052962,0.041586,"PCR_Ridge, PLS_Regression, Bayesian_Ridge, Hub..."
15,Vibrational response | FRF (g/N) | 6500-8000 H...,ok,,vibrational,robust_trimmed_median,WeightedBlend_2,raw,3,1,0.760744,0.000000,0.262866,0.124296,"PCR_Ridge, PLS_Regression, Bayesian_Ridge, Hub..."
8,Thermal characteristics | h | W/m.K,ok,,thermal,robust_trimmed_median,Kernel_Ridge_RBF,yeo_johnson,3,1,0.662518,0.000000,0.692994,0.622197,"Huber_Regressor, Bayesian_Ridge, PCR_Ridge, PL..."
5,Densif. strength,ok,,mechanical_energy,hard_closest_oof,PCR_Ridge,raw,4,3,0.645503,0.212854,153.183371,128.730794,"PCR_Ridge, PLS_Regression, Bayesian_Ridge, Rid..."
4,Yield strength,ok,,mechanical_energy,hard_closest_oof,ElasticNet_CV,raw,3,1,0.562975,0.000000,35.314399,28.130598,"PCR_Ridge, PLS_Regression, Bayesian_Ridge, Rid..."
7,Thermal characteristics | Thermal conductivity...,ok,,thermal,robust_trimmed_median,SVR_RBF,yeo_johnson,2,1,0.420380,0.000000,0.513086,0.400898,"Huber_Regressor, Bayesian_Ridge, PCR_Ridge, PL..."
0,Modulus,ok,,mechanical_energy,hard_closest_oof,Ridge,raw,2,2,0.239260,0.132486,2733.267283,2404.372710,"PCR_Ridge, PLS_Regression, Bayesian_Ridge, Rid..."
9,Thermal characteristics | Heating rate | °C/s,ok,,thermal,robust_trimmed_median,PCR_Ridge,raw,3,1,0.183450,0.000000,0.002570,0.001969,"Huber_Regressor, Bayesian_Ridge, PCR_Ridge, PL..."


In [8]:
# ============================================================
# Cell E1. Refit final model on full data + export final selected INPUT-OUTPUT datasets
# ============================================================

FINAL_REFIT_SUMMARY_ROWS = []
FINAL_MODEL_BACKUP = {}

for output_name, bundle in DATA_BY_OUTPUT.items():
    summary_hit = ALL_SUMMARY_DF[(ALL_SUMMARY_DF["output_name"] == output_name) & (ALL_SUMMARY_DF["status"] == "ok")]
    if summary_hit.empty:
        continue

    chosen_model_name = summary_hit.iloc[0]["final_model_name"]
    chosen_transform = summary_hit.iloc[0]["target_transform"]
    chosen_topk = int(summary_hit.iloc[0]["final_chosen_topk"])
    chosen_strategy = summary_hit.iloc[0]["best_y_strategy"]

    df = bundle["df"].copy()
    feature_cols = bundle["feature_cols"]
    target_col = bundle["target_col"]
    group_col = bundle["group_col"]
    xkey_col = bundle["xkey_col"]

    X_raw = df[feature_cols].copy()
    y_raw = pd.to_numeric(df[target_col], errors="coerce").to_numpy(dtype=float)
    g_raw = df[group_col].to_numpy()
    xkey_raw = df[xkey_col].to_numpy()

    rough_cols, rough_score_df = prefilter_features_groupwise(
        X_raw, y_raw, g_raw,
        top_k=get_prefilter_topk_for_output(output_name),
        corr_threshold=CORR_PRUNE_THRESHOLD,
        random_state=RANDOM_STATE + 500
    )
    if len(rough_cols) < 2:
        FINAL_REFIT_SUMMARY_ROWS.append({"output_name": output_name, "status": "failed", "reason": "rough prefilter failed"})
        continue

    X_raw_aug, added_cols_full = build_engineered_feature_space(X_raw, rough_cols, output_name=output_name)
    aug_feature_cols = list(X_raw_aug.columns)

    selected_full_df, candidate_full_df = select_best_y_within_group(
        X_raw_aug, y_raw, g_raw, xkey_raw,
        rough_cols=rough_cols,
        strategy=chosen_strategy,
        random_state=RANDOM_STATE + 600
    )

    X_train_sel = selected_full_df[aug_feature_cols].copy()
    y_train_sel = selected_full_df["target"].to_numpy(dtype=float)
    g_train_sel = selected_full_df["GROUP_ID"].to_numpy()

    final_prefilter_cols, final_score_df = prefilter_features_groupwise(
        X_train_sel, y_train_sel, g_train_sel,
        top_k=get_prefilter_topk_for_output(output_name),
        corr_threshold=CORR_PRUNE_THRESHOLD,
        random_state=RANDOM_STATE + 700
    )
    freq_df = build_feature_frequency(
        X_train_sel, y_train_sel, g_train_sel,
        candidate_cols=final_prefilter_cols,
        n_repeats=24,
        random_state=RANDOM_STATE + 800
    )

    rank_df = final_score_df[["feature_name", "ensemble_score"]].merge(freq_df, on="feature_name", how="left").fillna(0.0)
    rank_df["rank_score"] = 0.55 * minmax_series(rank_df["selection_frequency"]) + 0.45 * minmax_series(rank_df["ensemble_score"])
    rank_df = rank_df.sort_values("rank_score", ascending=False).reset_index(drop=True)
    ranked_features = rank_df["feature_name"].tolist()
    ranked_features = corr_prune_from_ranked(X_train_sel[ranked_features], ranked_features, keep_k=MAX_FINAL_FEATURES, threshold=CORR_PRUNE_THRESHOLD)

    fold_hit = ALL_FOLD_DF[
        (ALL_FOLD_DF["output_name"] == output_name)
        & (ALL_FOLD_DF["model_name"] == chosen_model_name)
        & (ALL_FOLD_DF["topk"] == chosen_topk)
        & (ALL_FOLD_DF["best_y_strategy"] == chosen_strategy)
    ].copy()

    consensus_features = []
    consensus_df = pd.DataFrame(columns=["feature_name", "count", "share"])
    if not fold_hit.empty:
        feat_counter = Counter()
        for txt in fold_hit["selected_features"].astype(str):
            feats = [t.strip() for t in txt.split(",") if t.strip()]
            for f in feats:
                feat_counter[f] += 1
        if len(feat_counter) > 0:
            consensus_df = pd.DataFrame({
                "feature_name": list(feat_counter.keys()),
                "count": list(feat_counter.values())
            }).sort_values(["count", "feature_name"], ascending=[False, True]).reset_index(drop=True)
            consensus_df["share"] = consensus_df["count"] / max(1, len(fold_hit))
            consensus_features = consensus_df.loc[consensus_df["share"] >= CONSENSUS_MIN_FOLD_SHARE, "feature_name"].tolist()
            ranked_features = reorder_ranked_with_consensus(ranked_features, consensus_features)

    best_features = ranked_features[:min(chosen_topk, len(ranked_features))]
    X_final = X_train_sel[best_features].copy()
    fit_info = fit_search_model(
        X_final, y_train_sel, g_train_sel,
        model_name=chosen_model_name,
        target_transform=chosen_transform,
        random_state=RANDOM_STATE + 900
    )
    final_estimator = fit_info["best_estimator"]
    final_pred = np.asarray(final_estimator.predict(X_final)).reshape(-1)

    export_feature_cols = list(dict.fromkeys(best_features + ["target", "GROUP_ID", "X_KEY"]))
    selected_export_df = selected_full_df[export_feature_cols].copy()
    for extra_col in ["model_aware_oof_pred", "abs_resid_to_oof", "group_median_y", "abs_to_group_median", "selected_strategy"]:
        if extra_col in selected_full_df.columns:
            selected_export_df[extra_col] = selected_full_df[extra_col].values
    selected_export_df.rename(columns={"target": output_name}, inplace=True)

    out_dir = os.path.join(FINAL_DATA_DIR, sanitize_filename(output_name))
    os.makedirs(out_dir, exist_ok=True)

    export_df(candidate_full_df, os.path.join(out_dir, "all_repeated_candidates_with_strategy"))
    export_df(selected_export_df, os.path.join(out_dir, "final_selected_input_output_dataset"))
    export_df(rank_df, os.path.join(out_dir, "final_feature_rank"))
    export_df(consensus_df, os.path.join(out_dir, "outer_fold_consensus_features"))

    backup_obj = {
        "output_name": output_name,
        "estimator": final_estimator,
        "selected_features": best_features,
        "selected_dataset": selected_export_df.copy(),
        "candidate_dataset": candidate_full_df.copy(),
        "feature_rank_df": rank_df.copy(),
        "consensus_feature_df": consensus_df.copy(),
        "consensus_features": list(consensus_features),
        "model_name": chosen_model_name,
        "target_transform": chosen_transform,
        "topk": int(chosen_topk),
        "best_y_strategy": chosen_strategy,
        "best_params": fit_info["best_params"],
        "train_r2": float(r2_score(y_train_sel, final_pred)),
        "train_rmse": float(math.sqrt(mean_squared_error(y_train_sel, final_pred))),
        "train_mae": float(mean_absolute_error(y_train_sel, final_pred)),
        "export_dir": out_dir,
        "engineered_added_cols": added_cols_full,
    }
    FINAL_MODEL_BACKUP[output_name] = backup_obj

    joblib.dump(backup_obj, os.path.join(FINAL_BACKUP_DIR, sanitize_filename(output_name) + "_model_backup.joblib"))

    FINAL_REFIT_SUMMARY_ROWS.append({
        "output_name": output_name,
        "status": "ok",
        "reason": "",
        "best_y_strategy": chosen_strategy,
        "final_model_name": chosen_model_name,
        "target_transform": chosen_transform,
        "final_chosen_topk": int(chosen_topk),
        "n_groups_full": int(len(pd.unique(g_train_sel))),
        "n_final_features": int(len(best_features)),
        "final_train_r2": float(r2_score(y_train_sel, final_pred)),
        "final_train_rmse": float(math.sqrt(mean_squared_error(y_train_sel, final_pred))),
        "final_train_mae": float(mean_absolute_error(y_train_sel, final_pred)),
    })

FINAL_REFIT_SUMMARY_DF = pd.DataFrame(FINAL_REFIT_SUMMARY_ROWS)
export_df(FINAL_REFIT_SUMMARY_DF, os.path.join(FINAL_DATA_DIR, "final_refit_summary"))
display(FINAL_REFIT_SUMMARY_DF.sort_values("final_train_r2", ascending=False))


ValueError: Unknown model: WeightedBlend_2

In [ ]:
# ============================================================
# Cell F1. Final merged summary
# ============================================================

FINAL_SUMMARY_DF = ALL_SUMMARY_DF.merge(
    FINAL_REFIT_SUMMARY_DF,
    on="output_name",
    how="left",
    suffixes=("_cv", "_refit")
)

export_df(FINAL_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "final_summary_merged"))
display(FINAL_SUMMARY_DF.sort_values("cv_r2_mean", ascending=False))

In [ ]:
# ============================================================
# Cell G1. Backup reload for later plotting / next cell usage
# ============================================================

# 이 셀을 실행하면 언제든지 저장된 모델/데이터를 다시 불러와서
# 다음 셀에서 scatter, parity plot, PDP 유사 그래프 등을 바로 그릴 수 있음.

BACKUP_FILES = sorted(Path(FINAL_BACKUP_DIR).glob("*_model_backup.joblib"))
AVAILABLE_OUTPUTS = [p.name.replace("_model_backup.joblib", "") for p in BACKUP_FILES]

print("Available backups:")
for name in AVAILABLE_OUTPUTS:
    print(" -", name)

# 예시: 첫 번째 output 자동 선택
if len(BACKUP_FILES) > 0:
    EXAMPLE_BACKUP_PATH = str(BACKUP_FILES[0])
    EXAMPLE_BACKUP = joblib.load(EXAMPLE_BACKUP_PATH)
    print("\nLoaded example backup:", EXAMPLE_BACKUP["output_name"])
    print("Model             :", EXAMPLE_BACKUP["model_name"])
    print("Target transform  :", EXAMPLE_BACKUP["target_transform"])
    print("Selected features :", EXAMPLE_BACKUP["selected_features"])
    print("Dataset shape     :", EXAMPLE_BACKUP["selected_dataset"].shape)

# 다음 셀에서 사용할 기본 예시:
# bk = joblib.load(os.path.join(FINAL_BACKUP_DIR, sanitize_filename("Modulus") + "_model_backup.joblib"))
# model = bk["estimator"]
# df_plot = bk["selected_dataset"].copy()
# feats = bk["selected_features"]
# y_col = bk["output_name"]
# y_true = df_plot[y_col].to_numpy(dtype=float)
# y_pred = np.asarray(model.predict(df_plot[feats])).reshape(-1)